In [ ]:
import ast
import json
import os
import random
import re
import time
import typing
from datetime import datetime
from typing import Dict, List, Optional, Set, Tuple, Union

import gseapy as gp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import torch
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer

In [ ]:
# =====================================================
# Load files
# =====================================================

validation = pd.read_csv("AML Validation V1.csv")
paragraph = pd.read_csv("Pathway_Annotation_ParagraphV1.csv")

# =====================================================
# Extract clean process name from paragraph file
# =====================================================

def extract_process_name(text):

    if pd.isna(text):
        return None

    text = str(text).strip()
    if "confidence score" not in text.lower():
        return text
    text = text.replace("**", " ")
    match = re.search(
        r'Name(?:\s+with|\s+and)?\s*&?\s*Confidence\s*Score\s*(.+)',
        text,
        flags=re.IGNORECASE
    )
    if not match:
        return text

    process = match.group(1).strip()
    process = re.sub(
        r'^Process\s*:\s*',
        '',
        process,
        flags=re.IGNORECASE
    )
    process = re.split(
        r'Confidence\s*Score',
        process,
        flags=re.IGNORECASE
    )[0]

    process = re.sub(r'\s+', ' ', process).strip()

    return process if process else None
paragraph["Final_Process_Paragraph"] = (
    paragraph["Final_Process"]
    .apply(extract_process_name)
)
paragraph["Final_Process_Paragraph"] = (
    paragraph["Final_Process"]
    .apply(extract_process_name)
)
def normalize_validation_genes(gene_string):
    try:
        genes = ast.literal_eval(gene_string)

        genes = [
            str(g).strip().upper()
            for g in genes
        ]

        genes = sorted(set(genes))

        return ",".join(genes)

    except Exception:
        return None


def normalize_paragraph_genes(gene_string):

    genes = [
        g.strip().upper()
        for g in str(gene_string).split(",")
    ]

    genes = [g for g in genes if g]

    genes = sorted(set(genes))

    return ",".join(genes)


validation["Gene_Key"] = (
    validation["Genes"]
    .apply(normalize_validation_genes)
)

paragraph["Gene_Key"] = (
    paragraph["Genes_String"]
    .apply(normalize_paragraph_genes)
)
def choose_process(row):

    if (
        row["Confidence_With_Enrichment_Before"]
        >=
        row["Confidence_Without_Enrichment_Before"]
    ):
        return row["Process_With_Enrichment_Original"]

    return row["Process_Without_Enrichment_Original"]


validation["Final_Process_Before_Validation"] = (
    validation.apply(
        choose_process,
        axis=1
    )
)

validation_subset = validation[
    [
        "Gene_Key",
        "GeneSet_Name",
        "Final_Process_Before_Validation"
    ]
].rename(
    columns={
        "GeneSet_Name": "Geneset"
    }
)

paragraph_subset = paragraph[
    [
        "Gene_Key",
        "Final_Process_Paragraph"
    ]
]
merged = validation_subset.merge(
    paragraph_subset,
    on="Gene_Key",
    how="inner"
)
final_output = merged[
    [
        "Gene_Key",
        "Geneset",
        "Final_Process_Before_Validation",
        "Final_Process_Paragraph"
    ]
].rename(
    columns={
        "Gene_Key": "Genes"
    }
)

final_output.to_csv(
    "AML_Validation_Paragraph_Merged.csv",
    index=False
)

print(f"Merged rows: {len(final_output)}")

# sanity check
print(final_output.head())

In [ ]:
with pd.option_context('display.max_rows', None):
    print(paragraph["Final_Process"])

In [ ]:
df_merged = final_output
df_merged

In [ ]:
df_final_subset = df_merged[
    [
        "Geneset",
        "Genes",
        "Final_Process_Before_Validation",
        "Final_Process_Paragraph",
    ]
]

In [ ]:
msigdb_df = pd.read_csv('/Users/justin.seby/Documents/venv/Justin/Karolinska Institutet/DDLS/DDLS Code/DDLS Projects/AML/Code/Final Versions of Code/Benchmark_SOTA_annotation_models/AML_msigdb_descriptions.csv')
df_final_subset = df_final_subset.merge(msigdb_df, on='Geneset')

In [ ]:
df_final_subset

In [ ]:
def run_ladder_validation_final(
    df,
    MODELS=None,
    COLORS=None,
    METHOD_NAMES=None,
    out_dir="ladder_validation_outputs_final",
    csv_name="validation_results_final_2methods.csv",
    winfig_name="Figure_WinCounts_Final",
    simfig_name="Figure_Similarity_Final",
    max_length=512,
    device=None,
    run_name=None,
    add_timestamp=False
):

    # ---------------- filename prefix ----------------
    prefix = f"{run_name}_" if run_name else ""
    if add_timestamp:
        from datetime import datetime
        ts = datetime.now().strftime("%Y%m%dT%H%M%S")
        prefix = f"{prefix}{ts}_"

    # ---------------- defaults ----------------
    if MODELS is None:
        MODELS = {
            "BioLORD-2023": "FremyCompany/BioLORD-2023",
            "MedCPT": "ncbi/MedCPT-Query-Encoder",
        }

    if METHOD_NAMES is None:
        METHOD_NAMES = {
            "Scaffold": "Final Scaffold",
            "Paragraph": "Final Paragraph",
        }

    if COLORS is None:
        COLORS = {
            "Final Scaffold": "#3498db",
            "Final Paragraph": "#2ecc71",
        }

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}\n")

    os.makedirs(out_dir, exist_ok=True)

    csv_final = os.path.join(out_dir, f"{prefix}{csv_name}")
    win_png = os.path.join(out_dir, f"{prefix}{winfig_name}.png")
    win_svg = os.path.join(out_dir, f"{prefix}{winfig_name}.svg")
    sim_png = os.path.join(out_dir, f"{prefix}{simfig_name}.png")
    sim_svg = os.path.join(out_dir, f"{prefix}{simfig_name}.svg")

    # ---------------- helpers ----------------
    def load_model(model_id):
        tok = AutoTokenizer.from_pretrained(model_id)
        mod = AutoModel.from_pretrained(model_id).to(device).eval()
        return tok, mod

    def get_embedding(text, tok, mod):
        if not isinstance(text, str) or not text.strip():
            return np.zeros(mod.config.hidden_size, dtype=float)
        inputs = tok(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
            padding=True
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            out = mod(**inputs)
        return out.last_hidden_state[:, 0, :].cpu().numpy().flatten()

    # ---------------- validation ----------------
    def validate_with_model(df_local, display_name, model_id):
        print(f"Validating: {display_name}")
        tok, mod = load_model(model_id)
        rows = []

        for _, row in tqdm(df_local.iterrows(), total=len(df_local), leave=False):
            ref = get_embedding(row.get("MSigDB_Brief_Description", ""), tok, mod)

            emb_scaf = get_embedding(row.get("Final_Process_Before_Validation", ""), tok, mod)
            emb_para = get_embedding(row.get("Final_Process_Paragraph", ""), tok, mod)

            eps = 1e-12
            sims = {
                "Scaffold": cosine_similarity(
                    ref.reshape(1, -1) + eps,
                    emb_scaf.reshape(1, -1) + eps
                )[0][0],
                "Paragraph": cosine_similarity(
                    ref.reshape(1, -1) + eps,
                    emb_para.reshape(1, -1) + eps
                )[0][0],
            }

            max_sim = max(sims.values())
            winners = [k for k, v in sims.items() if np.isclose(v, max_sim, atol=1e-6)]

            for k, sim in sims.items():
                rows.append({
                    "Geneset": row.get("Geneset"),
                    "Model": display_name,
                    "Method": METHOD_NAMES[k],
                    "Similarity": sim,
                    "Winner": k in winners,
                })

        del tok, mod
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return pd.DataFrame(rows)

    # ---------------- run ----------------
    all_results = []
    for name, path in MODELS.items():
        all_results.append(validate_with_model(df, name, path))

    results_df = pd.concat(all_results, ignore_index=True)
    results_df.to_csv(csv_final, index=False)
    print(f"✓ Saved: {csv_final}\n")

    # ================= VISUALIZATION =================
    plt.style.use("seaborn-v0_8-whitegrid")
    plt.rcParams.update({
        "figure.dpi": 150,
        "font.family": "Arial",
        "axes.titlesize": 13,
        "axes.titleweight": "bold",
        "axes.labelsize": 11,
        "svg.fonttype": "none"
    })

    methods = list(METHOD_NAMES.values())
    models_list = sorted(results_df["Model"].unique())
    x = np.arange(len(models_list))
    width = 0.35

    # -------- Win counts --------
    fig, ax = plt.subplots(figsize=(9, 5))

    for i, method in enumerate(methods):
        wins = [
            len(results_df[
                (results_df["Model"] == m) &
                (results_df["Method"] == method) &
                (results_df["Winner"])
            ])
            for m in models_list
        ]
        offset = (i - 0.5) * width
        bars = ax.bar(
            x + offset, wins, width,
            label=method,
            color=COLORS[method],
            edgecolor="black",
            linewidth=0.7,
            alpha=0.9
        )
        for bar, w in zip(bars, wins):
            if w > 0:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 1,
                    str(w),
                    ha="center",
                    va="bottom",
                    fontsize=9,
                    fontweight="bold"
                )

    ax.set_xticks(x)
    ax.set_xticklabels(models_list, rotation=30, ha="right")
    ax.set_ylabel("Number of Wins")
    ax.set_title("Final Annotation Comparison")
    ax.legend(frameon=True)
    sns.despine()
    plt.tight_layout()
    plt.savefig(win_png, dpi=600, bbox_inches="tight", facecolor="white")
    plt.savefig(win_svg, bbox_inches="tight", facecolor="white")
    plt.show()

    # -------- Raincloud similarity --------
    fig, ax = plt.subplots(figsize=(12, 6))

    model_positions = {m: i for i, m in enumerate(models_list)}
    offsets = np.linspace(-0.2, 0.2, len(methods))
    method_offsets = dict(zip(methods, offsets))

    for method in methods:
        color = COLORS[method]
        for model in models_list:
            vals = results_df[
                (results_df["Model"] == model) &
                (results_df["Method"] == method)
            ]["Similarity"].values
            if len(vals) == 0:
                continue

            x_pos = model_positions[model] + method_offsets[method]

            parts = ax.violinplot(
                [vals],
                positions=[x_pos],
                widths=0.15,
                showmeans=False,
                showmedians=False,
                showextrema=False
            )
            for pc in parts["bodies"]:
                pc.set_facecolor(color)
                pc.set_alpha(0.7)
                pc.set_edgecolor("black")

            ax.boxplot(
                [vals],
                positions=[x_pos + 0.07],
                widths=0.05,
                showfliers=False,
                patch_artist=True,
                boxprops=dict(facecolor=color, alpha=0.5),
                medianprops=dict(color="black")
            )

            jitter = x_pos + 0.14 + np.random.uniform(-0.02, 0.02, size=len(vals))
            ax.scatter(
                jitter,
                vals,
                color=color,
                alpha=0.4,
                s=20,
                edgecolor="black",
                linewidth=0.3
            )

    ax.set_xticks(list(model_positions.values()))
    ax.set_xticklabels(models_list, rotation=30, ha="right")
    ax.set_ylabel("Cosine Similarity")
    ax.set_title("Distribution of Final Annotation Similarity Scores")
    sns.despine()
    plt.tight_layout()
    plt.savefig(sim_png, dpi=600, bbox_inches="tight", facecolor="white")
    plt.savefig(sim_svg, bbox_inches="tight", facecolor="white")
    plt.show()

    print(f"✓ Figures saved:\n  {win_png}\n  {win_svg}\n  {sim_png}\n  {sim_svg}")

    return results_df

In [ ]:
invalid_values = {
    "name & confidence score",
    "name with confidence score",
    "name and confidence score",
    "process"
}

df_final_subset = df_final_subset[
    ~df_final_subset["Final_Process_Paragraph"]
        .fillna("")
        .str.strip()
        .str.lower()
        .isin(invalid_values)
]

In [ ]:
df_final_subset.to_csv(
    "AML_Validation_Paragraph_MergedV1.csv",
    index=False
)

In [ ]:
df_final_subset["Final_Process_Before_Validation"] = (
    df_final_subset["Final_Process_Before_Validation"]
    .str.replace(r"\bacute myeloid leukemia\b", "", case=False, regex=True)
    .str.replace(r"\baml\b", "", case=False, regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
df_final_subset["Final_Process_Paragraph"] = (
    df_final_subset["Final_Process_Paragraph"]
    .str.replace(r"\bacute myeloid leukemia\b", "", case=False, regex=True)
    .str.replace(r"\baml\b", "", case=False, regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)


In [ ]:
results_df = run_ladder_validation_final(df_final_subset, run_name="AMLScaffoldcompare", add_timestamp=False)